<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-05-bigquery-ml/lesson-5.1-bqml/practice/GCP_Capstone_5.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 5.1 — ML in SQL — CREATE MODEL

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup — auth, client, helpers & sample data

Run this first. It authenticates with Application Default Credentials, creates the BigQuery client, defines the `run_query` / `run_ddl` helpers every exercise uses, and builds the `rag_data.document_features` sample table plus the `ml_models` dataset. Everything below depends on it.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE to your project id

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

def run_query(sql):
    """Run a BQ query and return results as DataFrame."""
    return client.query(sql).to_dataframe()

def run_ddl(sql):
    """Run a DDL/DML statement."""
    job = client.query(sql)
    job.result()
    print(f'Done: {job.num_dml_affected_rows or "model created"}')

USD_INR = 85  # for INR cost displays
print(f'Connected to {PROJECT_ID}')

In [ ]:
# Datasets
run_ddl(f'CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.ml_models`')
run_ddl(f'CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.rag_data`')

# Sample document features table
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.document_features` AS
SELECT * FROM UNNEST([
  STRUCT('d1' AS doc_id, 'Q4 Report' AS title, 'pdf' AS content_type, 12 AS page_count, 2.5 AS file_size_mb, 24 AS chunk_count, 8500 AS total_word_count, 354.0 AS avg_chunk_size, FALSE AS has_tables, TRUE AS has_images, 0.018 AS processing_cost_usd, 'research_paper' AS document_type),
  STRUCT('d2','Invoice 2024-001','pdf',2,0.3,4,800,200.0,TRUE,FALSE,0.003,'invoice'),
  STRUCT('d3','NDA Agreement','pdf',8,1.1,16,4200,262.0,FALSE,FALSE,0.012,'legal'),
  STRUCT('d4','KYC Form','pdf',1,0.2,2,400,200.0,TRUE,FALSE,0.002,'form'),
  STRUCT('d5','ML Survey 2025','pdf',45,8.2,90,32000,355.0,TRUE,TRUE,0.068,'research_paper'),
  STRUCT('d6','Receipt Mar','pdf',1,0.1,2,250,125.0,TRUE,FALSE,0.002,'invoice'),
  STRUCT('d7','License Agreement','pdf',15,2.8,30,12000,400.0,FALSE,FALSE,0.023,'legal'),
  STRUCT('d8','Employee Onboarding','pdf',3,0.5,6,1800,300.0,TRUE,FALSE,0.005,'form'),
  STRUCT('d9','RAG Architecture','pdf',22,4.1,44,16000,363.0,TRUE,TRUE,0.033,'research_paper'),
  STRUCT('d10','GST Invoice','pdf',1,0.15,2,350,175.0,TRUE,FALSE,0.002,'invoice'),
  STRUCT('d11','Partnership Deed','pdf',20,3.5,40,15000,375.0,FALSE,FALSE,0.030,'legal'),
  STRUCT('d12','Leave Application','pdf',1,0.1,2,300,150.0,TRUE,FALSE,0.002,'form')
])
''')
print('Sample data created')
run_query(f'SELECT doc_id, title, document_type, processing_cost_usd FROM `{PROJECT_ID}.rag_data.document_features`')

## Exercise 1: First CREATE MODEL

**Difficulty:** Easy

Train a LINEAR_REG model predicting processing cost. Check ML.TRAINING_INFO for loss curve.

1. CREATE MODEL with model_type='LINEAR_REG'
2. Set input_label_cols, max_iterations, early_stop
3. Query ML.TRAINING_INFO for loss per iteration

In [ ]:
# Train the linear regression cost predictor
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.cost_predictor`
OPTIONS (
  model_type = 'LINEAR_REG',
  input_label_cols = ['processing_cost_usd'],
  max_iterations = 20,
  early_stop = TRUE,
  l2_reg = 0.1,
  enable_global_explain = TRUE
) AS
SELECT
  page_count, chunk_count, total_word_count,
  file_size_mb, content_type, processing_cost_usd
FROM `{PROJECT_ID}.rag_data.document_features`
WHERE processing_cost_usd IS NOT NULL
''')
print('Cost predictor trained')

# Inspect the loss curve — should decrease across iterations
print('\n=== Training Progress ===')
print(run_query(f'''
SELECT iteration, training_loss, eval_loss, duration_ms
FROM ML.TRAINING_INFO(MODEL `{PROJECT_ID}.ml_models.cost_predictor`)
ORDER BY iteration
'''))

## Exercise 2: ML.EVALUATE + ML.PREDICT

**Difficulty:** Easy

Evaluate your regression model. Predict costs for documents. Print r2_score.

1. ML.EVALUATE returns r2_score, MAE, MSE
2. ML.PREDICT returns predicted_processing_cost_usd
3. Interpret: r2 > 0.7 = good, < 0.5 = needs more features

In [ ]:
# Evaluate
print('=== Regression Metrics ===')
metrics = run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.cost_predictor`)')
print(metrics)
print(f"\nr2_score = {metrics['r2_score'].iloc[0]:.3f}")

# Predict
print('\n=== Predictions ===')
print(run_query(f'''
SELECT doc_id, title,
  ROUND(predicted_processing_cost_usd, 4) AS predicted_cost
FROM ML.PREDICT(
  MODEL `{PROJECT_ID}.ml_models.cost_predictor`,
  (SELECT * FROM `{PROJECT_ID}.rag_data.document_features`))
'''))

## Exercise 3: Feature Importance

**Difficulty:** Easy

Run ML.GLOBAL_EXPLAIN. Identify the top 3 features driving processing cost.

1. Requires enable_global_explain=TRUE at creation
2. Query ML.GLOBAL_EXPLAIN
3. Rank features by attribution_score

In [ ]:
# The cost_predictor was created with enable_global_explain = TRUE (Exercise 1),
# so global feature attributions are available.
print('=== Feature Importance (ranked) ===')
print(run_query(f'''
SELECT feature, attribution
FROM ML.GLOBAL_EXPLAIN(MODEL `{PROJECT_ID}.ml_models.cost_predictor`)
ORDER BY attribution DESC
LIMIT 3
'''))

## Exercise 4: Document Classifier

**Difficulty:** Medium

Train LOGISTIC_REG with auto_class_weights. Evaluate with ML.CONFUSION_MATRIX.

1. CREATE MODEL with LOGISTIC_REG
2. Set auto_class_weights=TRUE
3. Run ML.CONFUSION_MATRIX to see misclassifications

In [ ]:
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.doc_classifier`
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['document_type'],
  auto_class_weights = TRUE,
  max_iterations = 20,
  enable_global_explain = TRUE
) AS
SELECT
  page_count, total_word_count, avg_chunk_size,
  chunk_count, file_size_mb, has_tables, has_images,
  document_type
FROM `{PROJECT_ID}.rag_data.document_features`
''')
print('Document classifier trained')

# Evaluate
print('\n=== Classification Metrics ===')
print(run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.doc_classifier`)'))

# Confusion matrix — high diagonal = correct classifications
print('\n=== Confusion Matrix ===')
print(run_query(f'SELECT * FROM ML.CONFUSION_MATRIX(MODEL `{PROJECT_ID}.ml_models.doc_classifier`)'))

## Exercise 5: K-Means on Embeddings

**Difficulty:** Medium

Cluster chunks with COSINE distance. Assign CENTROID_IDs. Inspect cluster contents.

1. CREATE MODEL with KMEANS, COSINE, standardize=FALSE
2. ML.PREDICT to get CENTROID_ID per chunk
3. Group by CENTROID_ID, inspect content previews

In [ ]:
# Synthetic embeddings (in production, use real embeddings from Lesson 2.2)
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.chunk_embeddings` AS
SELECT
  CONCAT('chunk_', CAST(n AS STRING)) AS chunk_id,
  CONCAT('doc_', CAST(MOD(n, 5) + 1 AS STRING)) AS doc_id,
  ARRAY(SELECT RAND() FROM UNNEST(GENERATE_ARRAY(1, 10))) AS embedding
FROM UNNEST(GENERATE_ARRAY(1, 50)) AS n
''')

# Train K-Means with COSINE distance, no standardization
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.topic_clusters`
OPTIONS (
  model_type = 'KMEANS',
  num_clusters = 5,
  kmeans_init_method = 'KMEANS++',
  distance_type = 'COSINE',
  standardize_features = FALSE,
  max_iterations = 50
) AS
SELECT embedding
FROM `{PROJECT_ID}.rag_data.chunk_embeddings`
WHERE embedding IS NOT NULL
''')
print('Topic clusters trained')

# Evaluate
print('\n=== Clustering Metrics ===')
print(run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.topic_clusters`)'))

# Assign CENTROID_ID per chunk, grouped by cluster
print('\n=== Cluster Assignments ===')
print(run_query(f'''
SELECT chunk_id, doc_id, CENTROID_ID AS topic_cluster
FROM ML.PREDICT(
  MODEL `{PROJECT_ID}.ml_models.topic_clusters`,
  (SELECT * FROM `{PROJECT_ID}.rag_data.chunk_embeddings`))
ORDER BY topic_cluster
LIMIT 20
'''))

## Exercise 6: Optimal K Comparison

**Difficulty:** Medium

Train k=5,10,15,20. Compare Davies-Bouldin scores. Select the best k.

1. Create 4 models with different num_clusters
2. ML.EVALUATE each, extract davies_bouldin_index
3. UNION ALL to compare in one result

In [ ]:
# Train one K-Means model per candidate k on the same embeddings
for k in [5, 10, 15, 20]:
    run_ddl(f'''
    CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.topic_clusters_k{k}`
    OPTIONS (
      model_type = 'KMEANS',
      num_clusters = {k},
      kmeans_init_method = 'KMEANS++',
      distance_type = 'COSINE',
      standardize_features = FALSE,
      max_iterations = 50
    ) AS
    SELECT embedding
    FROM `{PROJECT_ID}.rag_data.chunk_embeddings`
    WHERE embedding IS NOT NULL
    ''')
    print(f'Trained topic_clusters_k{k}')

# Compare Davies-Bouldin index across all k in one result (lower = better separation)
union_sql = '\nUNION ALL\n'.join([
    f"SELECT {k} AS num_clusters, davies_bouldin_index "
    f"FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.topic_clusters_k{k}`)"
    for k in [5, 10, 15, 20]
])
print('\n=== Davies-Bouldin by k (lowest wins) ===')
print(run_query(f'SELECT * FROM ({union_sql}) ORDER BY davies_bouldin_index'))

## Exercise 7: Boosted Tree Upgrade

**Difficulty:** Challenge

Upgrade LOGISTIC_REG to BOOSTED_TREE_CLASSIFIER with TRANSFORM. Compare f1 scores.

1. Add TRANSFORM with STANDARD_SCALER, QUANTILE_BUCKETIZE
2. CREATE MODEL with BOOSTED_TREE_CLASSIFIER
3. Compare ML.EVALUATE metrics side by side

In [ ]:
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.doc_classifier_boosted`
TRANSFORM (
  page_count,
  ML.STANDARD_SCALER(total_word_count) OVER() AS scaled_words,
  ML.QUANTILE_BUCKETIZE(file_size_mb, 5) OVER() AS size_bucket,
  chunk_count, avg_chunk_size, has_tables, has_images,
  document_type
)
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['document_type'],
  max_tree_depth = 4,
  max_iterations = 30,
  learn_rate = 0.1,
  subsample = 0.8,
  early_stop = TRUE,
  enable_global_explain = TRUE
) AS
SELECT * FROM `{PROJECT_ID}.rag_data.document_features`
''')
print('Boosted tree classifier trained')

# Compare f1 side by side
lr = run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.doc_classifier`)')
bt = run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.ml_models.doc_classifier_boosted`)')
print('\n=== Logistic Reg Metrics ===')
print(lr)
print('\n=== Boosted Tree Metrics ===')
print(bt)
print(f"\nf1: logistic={lr['f1_score'].iloc[0]:.3f}  boosted={bt['f1_score'].iloc[0]:.3f}")

## Exercise 8: 3-Model Pipeline

**Difficulty:** Challenge

Run all 3 ML.PREDICT calls: cost estimator, doc classifier, topic clusters on new data.

1. ML.PREDICT with cost_predictor for budget estimation
2. ML.PREDICT with doc_classifier for auto-tagging
3. ML.PREDICT with topic_clusters for chunk labeling

In [ ]:
# 1. Cost estimation — sum predicted cost across the incoming batch and show INR budget
cost = run_query(f'''
SELECT
  ROUND(SUM(predicted_processing_cost_usd), 4) AS total_cost_usd
FROM ML.PREDICT(
  MODEL `{PROJECT_ID}.ml_models.cost_predictor`,
  (SELECT * FROM `{PROJECT_ID}.rag_data.document_features`))
''')
total_usd = float(cost['total_cost_usd'].iloc[0])
print('=== 1. Budget Estimation (cost_predictor) ===')
print(f'Batch processing cost: ${total_usd:.4f}  (Rs {total_usd * USD_INR:.2f})')

# 2. Auto-tagging — predicted document_type per doc
print('\n=== 2. Auto-Tagging (doc_classifier) ===')
print(run_query(f'''
SELECT doc_id, title, predicted_document_type
FROM ML.PREDICT(
  MODEL `{PROJECT_ID}.ml_models.doc_classifier`,
  (SELECT * FROM `{PROJECT_ID}.rag_data.document_features`))
'''))

# 3. Chunk labeling — cluster id per chunk
print('\n=== 3. Chunk Labeling (topic_clusters) ===')
print(run_query(f'''
SELECT chunk_id, doc_id, CENTROID_ID AS topic_cluster
FROM ML.PREDICT(
  MODEL `{PROJECT_ID}.ml_models.topic_clusters`,
  (SELECT * FROM `{PROJECT_ID}.rag_data.chunk_embeddings`))
ORDER BY topic_cluster
LIMIT 15
'''))